## Setup

Loads the myocardial mask dataset, defines the CVAE architecture and shared evaluation utilities used by both the deployment section and the systematic latent-dimension evaluation below.

In [ ]:
from IPython import display

import glob
import imageio
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
import tensorflow_probability as tfp
import time
import pandas as pd
import tifffile
import cv2

from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#This is loading data, above lines are to load from google drive

import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Directory containing your images
image_folder = './data/SAM_masks_10000'

# List all TIFF files in the folder
image_files = sorted([f for f in os.listdir(image_folder) if f.endswith('.tiff')])

#load OG MRI mean T1 file
mean_T1_original = pd.read_csv('./data/mean_T1_originalfile.csv')
valid_ids = mean_T1_original.loc[mean_T1_original['quality_controlled'], 'id'].astype(str).tolist()

# Load, resize, and normalize images
images = []
for file in image_files:
    file_id = file[:7]  # First 7 characters of the file name
    if file_id in valid_ids:
        image_path = os.path.join(image_folder, file)

        # Load image in grayscale,  and convert to numpy array
        img = tifffile.imread(image_path)  # 'L' mode converts to grayscale
        img_array = np.array(img, dtype=np.float32) / 4000.0  # Normalize to [0,1]

        # Append to list
        images.append(img_array)

# Convert list of images to a single numpy array
images = np.array(images)


In [ ]:
# Split the data into 80% train and 10% test and 10% validation
train_images, test_images = train_test_split(images, test_size=0.2, random_state=42)
val_images, test_images = train_test_split(test_images, test_size=0.5, random_state=42)

# Check the shapes of the resulting arrays
print("x_train shape:", train_images.shape)  # Expected: (num_train_samples, 256, 256)
print("x_val shape:", val_images.shape)    # Expected: (num_test_samples, 256, 256)
print("x_test shape:", test_images.shape)    # Expected: (num_test_samples, 256, 256)

In [ ]:
def get_bounding_box(ground_truth_map):
  # get bounding box from mask
  y_indices, x_indices = np.where(ground_truth_map > 0)
  x_min, x_max = np.min(x_indices), np.max(x_indices)
  y_min, y_max = np.min(y_indices), np.max(y_indices)
  # add perturbation to bounding box coordinates
  H, W = ground_truth_map.shape
  x_min = max(0, x_min - 5)
  x_max = min(W, x_max + 5)
  y_min = max(0, y_min - 5)
  y_max = min(H, y_max + 5)
  bbox = [x_min, y_min, x_max, y_max]

  return bbox

def preprocess_images(images):
  filtered_images = []
  for image in images:
    try:
      bbox = get_bounding_box(image)
      image = image[bbox[1]:bbox[3], bbox[0]:bbox[2]]
      image = cv2.resize(image, (256, 256))
      filtered_images.append(image)
    except:
      continue
  filtered_images = np.array(filtered_images)
  filtered_images = filtered_images.reshape((filtered_images.shape[0], 256, 256, 1)) #/ 255.

  return np.array(filtered_images)

train_images = preprocess_images(train_images)
test_images = preprocess_images(test_images)
val_images = preprocess_images(val_images)


In [ ]:
def plot_distribution(image, threshold):
    non_zero_values = image[image > threshold]

    # Plot the distribution
    plt.figure(figsize=(8, 6))
    plt.hist(non_zero_values, bins=50, color='blue', alpha=0.7, edgecolor='black')
    plt.title("Distribution of Non-Zero Pixel Values", fontsize=14)
    plt.xlabel("Pixel Intensity", fontsize=12)
    #plt.xlim(-100, 2000)
    plt.ylabel("Frequency", fontsize=12)
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    return non_zero_values

plot_distribution(train_images[2] * 4000, 0)

In [ ]:
train_size = 4000
batch_size = 36
val_size = 500
test_size = 500

In [ ]:
train_dataset = (tf.data.Dataset.from_tensor_slices(train_images)
                 .shuffle(train_size).batch(batch_size))
test_dataset = (tf.data.Dataset.from_tensor_slices(test_images)
                .shuffle(test_size).batch(batch_size))
val_dataset = (tf.data.Dataset.from_tensor_slices(val_images)
                .shuffle(val_size).batch(batch_size))

In [ ]:
import tensorflow as tf

class CVAE(tf.keras.Model):
    """Convolutional Variational Autoencoder with input shape (256, 256, 1)."""

    def __init__(self, latent_dim):
        super(CVAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = tf.keras.Sequential(
            [
                tf.keras.layers.InputLayer(input_shape=(256, 256, 1)),
                tf.keras.layers.Conv2D(
                    filters=32, kernel_size=3, strides=(2, 2), activation='relu'),
                tf.keras.layers.Conv2D(
                    filters=64, kernel_size=3, strides=(2, 2), activation='relu'),
                tf.keras.layers.Conv2D(
                    filters=128, kernel_size=3, strides=(2, 2), activation='relu'),
                tf.keras.layers.Conv2D(
                    filters=256, kernel_size=3, strides=(2, 2), activation='relu'),
                tf.keras.layers.Flatten(),
                tf.keras.layers.Dense(latent_dim + latent_dim),
            ]
        )

        self.decoder = tf.keras.Sequential(
            [
                tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
                tf.keras.layers.Dense(units=16 * 16 * 256, activation=tf.nn.relu),
                tf.keras.layers.Reshape(target_shape=(16, 16, 256)),
                tf.keras.layers.Conv2DTranspose(
                    filters=256, kernel_size=4, strides=2, padding='same', activation='relu'),
                tf.keras.layers.Conv2DTranspose(
                    filters=128, kernel_size=4, strides=2, padding='same', activation='relu'),
                tf.keras.layers.Conv2DTranspose(
                    filters=64, kernel_size=4, strides=2, padding='same', activation='relu'),
                tf.keras.layers.Conv2DTranspose(
                    filters=1, kernel_size=4, strides=2, padding='same'),
            ]
        )
      #   self.discriminator = tf.keras.Sequential(
      #       [
      #           tf.keras.layers.InputLayer(input_shape=(256, 256, 1)),  # Input shape for 3-channel images
      #           tf.keras.layers.Conv2D(filters=32, kernel_size=5, strides=(1, 1), padding='same', activation='relu'),
      #           tf.keras.layers.Conv2D(filters=64, kernel_size=4, strides=(2, 2), padding='same', activation='relu'),  # Downsample
      #           tf.keras.layers.Conv2D(filters=128, kernel_size=4, strides=(2, 2), padding='same', activation='relu'),  # Downsample
      #           tf.keras.layers.Conv2D(filters=256, kernel_size=4, strides=(2, 2), padding='same', activation='relu'),  # Downsample
      #           tf.keras.layers.Conv2D(filters=256, kernel_size=4, strides=(2, 2), padding='same', activation='relu'),  # Downsample
      #           tf.keras.layers.Flatten(),  # Flatten for the fully connected layer
      #           tf.keras.layers.Dense(units=512, activation='relu', use_bias=False),  # Dense layer with 512 units
      #           tf.keras.layers.BatchNormalization(momentum=0.9),  # Batch normalization
      #           tf.keras.layers.ReLU(),  # ReLU activation
      #           tf.keras.layers.Dense(units=1),  # Final dense layer for the discriminator output
      #           tf.keras.layers.Activation('sigmoid'),  # Sigmoid for binary classification
      #       ]
      # )

    @tf.function
    def sample(self, eps=None):
        if eps is None:
            eps = tf.random.normal(shape=(100, self.latent_dim))
        return self.decode(eps, apply_sigmoid=True)

    def encode(self, x):
        mean, logvar = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=mean.shape)
        return eps * tf.exp(logvar * .5) + mean

    def decode(self, z, apply_sigmoid=False):
        logits = self.decoder(z)
        if apply_sigmoid:
            probs = tf.sigmoid(logits)
            return probs
        return logits

    # def discriminator(self, x):
    #     logits = self.discriminator(x)
    #     return logits

In [ ]:
from scipy.stats import ttest_ind, pearsonr

def post_processing(prediction, threshold):
    processed = np.where(prediction < threshold, 0, prediction)
    return processed

# Mean Squared Error (MSE)
def compute_mse_loss(predictions, ground_truth):
    # Compute the Mean Squared Error between predictions and ground truth
    return tf.reduce_mean(tf.square(predictions - ground_truth))

# PNSR: ratio between the maximum possible pixel value and the noise in the image
def compute_psnr(predictions, ground_truth):
    # PSNR formula: PSNR = 20 * log10(MAX_I) - 10 * log10(MSE)
    max_pixel_value = 1.0  # Assuming normalized pixel values (0 to 1)
    mse = np.mean(np.square(predictions - ground_truth))
    if mse == 0:
        return 100  # Return a high value when no error
    return 20 * np.log10(max_pixel_value / np.sqrt(mse))

# SSIM: structural similiarity index between luminance, contrast, and structure
def compute_ssim(predictions, ground_truth):
  ssim = tf.image.ssim(predictions, ground_truth, max_val=1.0)
  return ssim.numpy().tolist()

# Calculate statistics
def calculate_metrics(array):
    non_zero_values = array[array > 0]

    return {
        "mean": np.mean(non_zero_values),
        "median": np.median(non_zero_values),
        "std_dev": np.std(non_zero_values),
        "q25": np.percentile(non_zero_values, 25),
        "q75": np.percentile(non_zero_values, 75),
    }


# Deploy and Evaluate Model

In [ ]:
 # Deploying the model

model_path = './data/VAE_model_weights/cvae_16d_optimized.weights.h5'

latent_dim = 16
model = CVAE(latent_dim)
model.build(input_shape=(256, 256, 1))
model.load_weights(model_path)

In [ ]:
# Preprocess the input image (load in grayscale and normalize)
def preprocess_image(image_path, target_size=(256, 256)):
    # Load image in grayscale, resize, and normalize
    img = tifffile.imread(image_path)  # 'L' mode converts to grayscale
    img_array = np.array(img, dtype=np.float32) / 4000.0  # Normalize to [0,1]

    # Crop to the myocardium bounding box before resizing (matches the
    # production training pipeline's preprocessing)
    bbox = get_bounding_box(img_array)
    img_array = img_array[bbox[1]:bbox[3], bbox[0]:bbox[2]]
    img_array = cv2.resize(img_array, target_size)

    # Add a batch dimension (for compatibility with the model)
    img_array = np.expand_dims(img_array, axis=-1)  # Add channel dimension (1 for grayscale)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array

# Encode the image and get the latent space representation
def encode_image(model, image):
    mean, logvar = model.encode(image)  # Encoding step
    return mean.numpy()  # Return the mean (latent space representation)

def post_processing(prediction, threshold):
    processed = np.where(prediction < threshold, 0, prediction)
    return processed

# Process a directory of images and return a DataFrame with image names and latent spaces
def process_dataset(dataset_dir, model):
    latent_spaces = []
    image_names = []

    image_files = sorted([f for f in os.listdir(image_folder)])
    image_paths = [os.path.join(dataset_dir, fname) for fname in image_files]

    for image_path in image_paths:
        # Preprocess the input image
        image = preprocess_image(image_path)

        # Get the latent space encoding
        latent_space = encode_image(model, image)

        # Extract the filename without the directory path
        image_name = os.path.basename(image_path)
        image_names.append(image_name)

        # Append the latent space representation (flattened into a 1D array)
        latent_spaces.append(latent_space.flatten())

        # Optionally print the progress
        # print(f"Processed: {image_path}")

    # Convert the list of latent spaces to a pandas DataFrame
    latent_df = pd.DataFrame(latent_spaces)
    latent_df['IID'] = image_names  # Add a column for image filenames
    latent_df['IID'] = latent_df['IID'].str[:7]

    # Reorder columns to have 'image_name' first
    latent_df = latent_df[['IID'] + [col for col in latent_df.columns if col != 'IID']]

    return latent_df


In [ ]:
dataset_dir = './data/SAM_masks_10000'
image_files = sorted([f for f in os.listdir(dataset_dir)])
image_paths = [os.path.join(dataset_dir, fname) for fname in image_files]

example = image_paths[64]

example_array = preprocess_image(example)

mean, logvar = model.encode(example_array)
z = model.reparameterize(mean, logvar)
prediction = model.sample(z)

In [ ]:
# Raw Ground Truth
plt.subplot(1, 4, 1)
plt.imshow(np.array(tf.squeeze(example_array)) * 4000, cmap='coolwarm')
plt.title("Ground Truth")
plt.axis('off')

# Raw Predictions
plt.subplot(1, 3, 2)
plt.imshow(tf.squeeze(prediction) * 4000, cmap='coolwarm')
plt.title("Predictions Raw")
plt.axis('off')

prediction_2 = post_processing(tf.squeeze(prediction), 0.20)

# Processed Predictions
plt.subplot(1, 3, 3)
plt.imshow(prediction_2 * 4000, cmap='coolwarm')
plt.title("Predictions 0.2 cutoff")
plt.axis('off')

plt.show()

prediction_2.shape

In [ ]:
dataset_dir = './data/SAM_masks_10000'
latent_df = process_dataset(dataset_dir, model)

In [ ]:
latent_df.to_csv('./data/latent_df_contrasted.txt', sep=' ', index=False)

## Increase Contrast Functions

In [ ]:
from scipy.stats import norm

def change_contrast(image, factor):
    # Mask out pixels with value less than 0.0001
    mask = image >= 0.0001
    valid_pixels = image[mask]

    # Calculate the mean and standard deviation of valid pixels
    mean = np.mean(valid_pixels)
    std_dev = np.std(valid_pixels)

    # Define the new standard deviation (4 times the original)
    new_std_dev = factor * std_dev

    # Map each valid pixel to its percentile in the original distribution
    percentiles = norm.cdf(valid_pixels, loc=mean, scale=std_dev)

    # Transform percentiles to pixel values in the new distribution
    new_values = norm.ppf(percentiles, loc=mean, scale=new_std_dev)

    # Create the output image
    enhanced_image = np.copy(image)
    enhanced_image[mask] = new_values

    return enhanced_image

def histogram_equalization(image):
    mask = image > 0  # Select non-zero pixels
    valid_pixels = image[mask]

    # Normalize to 0-1
    normalized = (valid_pixels - valid_pixels.min()) / (valid_pixels.max() - valid_pixels.min())

    # Apply histogram equalization
    equalized = (normalized * 255).astype(np.uint8)
    result = np.copy(image)
    result[mask] = equalized
    return result

def gamma_correction(image, gamma):
    mask = image > 0  # Select non-zero pixels
    valid_pixels = image[mask]

    # Apply gamma correction
    corrected = np.power(valid_pixels, gamma)

    # Replace non-zero pixels
    result = np.copy(image)
    result[mask] = corrected
    return result

In [ ]:
image_example = train_images[5]

enhanced_image = gamma_correction(image_example, 5)

img_array = np.expand_dims(np.array(enhanced_image), axis=-1)  # Add channel dimension (1 for grayscale)
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

mean, logvar = encode_image(model, img_array)
z = model.reparameterize(mean, logvar)
prediction = model.sample(z)

enhanced_prediction = gamma_correction(prediction, 5)

In [ ]:
# Display the images
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(image_example, cmap="magma")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.title("Enhanced Image")
plt.imshow(enhanced_image, cmap="magma")
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
# Metrics on just the myocardium

# Calculate MSE for the test set
mse_myocardium_values = []

# Calculate PSNR for the test set
psnr_myocardium_values = []

# Calculate SSIM for the test set
ssim_myocardium_values = []

gamma = 5

def gamma_correction(image, gamma):
    # Create a mask for non-zero pixels
    mask = image > 0  # Boolean mask of non-zero pixels

    # Apply gamma correction to valid (non-zero) pixels only
    # Using tf.where to apply gamma correction to valid pixels, leave others unchanged
    corrected = tf.where(mask, tf.pow(image, gamma), image)

    return corrected

for test_x in val_dataset:
    # Get model predictions for the test set
    mean, logvar = model.encode(test_x)
    z = model.reparameterize(mean, logvar)
    predictions = model.sample(z)

    # Create a binary mask for non-zero pixels in the ground truth
    mask = tf.cast(test_x > 0, dtype=tf.float32)  # Binary mask for non-zero pixels

    # Apply the mask to the predictions to isolate the myocardium
    masked_predictions = predictions * mask
    masked_test_x = test_x * mask

    # Apply gamma correction to each batch (corrected only for non-zero pixels)
    corrected_test_x = gamma_correction(masked_test_x, gamma)
    corrected_predictions = gamma_correction(masked_predictions, gamma)

    # Compute the MSE between predictions and ground truth
    mse = compute_mse_loss(corrected_predictions, corrected_test_x)
    mse_myocardium_values.append(mse.numpy())

    # Compute PSNR between predictions and ground truth
    psnr = compute_psnr(corrected_predictions.numpy(), corrected_test_x.numpy())
    psnr_myocardium_values.append(psnr)

    # Compute SSIM between predictions and ground truth
    ssim = compute_ssim(corrected_predictions, corrected_test_x)
    ssim_myocardium_values.extend(ssim)

# Average MSE over the test set
average_mse = np.mean(mse_myocardium_values)
print(f'Average MSE on test set myocardium: {average_mse}')

# Average PSNR over the test set
average_psnr = np.mean(psnr_myocardium_values)
print(f'Average PSNR on test set myocardium: {average_psnr}')

# Average SSIM over the test set
average_ssim = np.mean(ssim_myocardium_values)
print(f'Average SSIM on test set myocardium: {average_ssim}')

### Evaluation of multiple dimensions across several models

In [ ]:
latent_dim = 6

model_path = f'./data/VAE_model_weights/cvae_{latent_dim}d_10000_boxbounded.weights.h5'

model = CVAE(latent_dim)
model.build(input_shape=(256, 256, 1))
model.load_weights(model_path)

In [ ]:
def generate_base_predictions(model):
    """Generate base predictions for the validation set."""
    means = []
    logvars = []
    zs = []
    predictions = []
    originals = []

    # Calculate MSE for the test set
    mse_values = []

    # Calculate PSNR for the test set
    psnr_values = []

    # Calculate SSIM for the test set
    ssim_values = []

    for test_x in tqdm(val_dataset, desc="Generating base predictions"):
        mean, logvar = model.encode(test_x)
        z = model.reparameterize(mean, logvar)
        prediction = model.sample(z)

        means.append(mean)
        logvars.append(logvar)
        zs.append(z)
        predictions.append(prediction)
        originals.append(test_x)

        # Compute the MSE between predictions and ground truth
        mse = compute_mse_loss(prediction, test_x)
        mse_values.append(np.mean(mse.numpy()))

        # Compute PSNR between predictions and ground truth
        psnr = compute_psnr(prediction, test_x.numpy())
        psnr_values.append(np.mean(psnr))

        # Compute SSIM between predictions and ground truth
        ssim = compute_ssim(prediction, test_x)
        ssim_values.append(np.mean(ssim))

    base_means = tf.concat(means, axis=0)
    base_logvars = tf.concat(logvars, axis=0)
    base_zs = tf.concat(zs, axis=0)
    base_predictions = tf.concat(predictions, axis=0)
    originals = tf.concat(originals, axis=0)


    return base_means, base_logvars, base_zs, base_predictions, originals, mse_values, psnr_values, ssim_values

In [ ]:
def analyze_dimension_importance(model, base_means, base_logvars, base_zs, base_predictions, originals):
    """Analyze importance of each latent dimension using multiple metrics."""
    n_dims = base_zs.shape[1]
    base_reconstruction_error = tf.reduce_mean(tf.square(originals - base_predictions))

    reconstruction_impact = []
    perturbation_sensitivity = []
    kl_divergence = []

    for dim in tqdm(range(n_dims), desc="Analyzing dimensions"):
        # Test reconstruction impact of zeroing out dimension
        modified_z = base_zs.numpy().copy()
        modified_z[:, dim] = 0
        modified_predictions = model.sample(modified_z)
        dim_impact = tf.reduce_mean(tf.square(originals - modified_predictions)) - base_reconstruction_error
        reconstruction_impact.append(float(dim_impact))

        # Test sensitivity to perturbations
        epsilon = 0.1
        perturbed_z = base_zs.numpy().copy()
        perturbed_z[:, dim] += epsilon
        perturbed_predictions = model.sample(perturbed_z)
        sensitivity = tf.reduce_mean(tf.abs(perturbed_predictions - base_predictions)) / epsilon
        perturbation_sensitivity.append(float(sensitivity))

        # Calculate KL divergence for this dimension
        mean = base_means[:, dim]
        logvar = base_logvars[:, dim]
        kl_div = -0.5 * tf.reduce_mean(1 + logvar - tf.square(mean) - tf.exp(logvar))
        kl_divergence.append(float(kl_div))

    # Normalize metrics
    reconstruction_impact = np.array(reconstruction_impact)
    reconstruction_impact = reconstruction_impact / np.max(reconstruction_impact)

    perturbation_sensitivity = np.array(perturbation_sensitivity)
    perturbation_sensitivity = perturbation_sensitivity / np.max(perturbation_sensitivity)

    kl_divergence = np.array(kl_divergence)
    kl_divergence = kl_divergence / np.max(kl_divergence)

    return reconstruction_impact, perturbation_sensitivity, kl_divergence

In [ ]:
base_means, base_logvars, base_zs, base_predictions, originals, mse_values, psnr_values, ssim_values = generate_base_predictions(model)

In [ ]:
reconstruction_impact, perturbation_sensitivity, kl_divergence = analyze_dimension_importance(model, base_means, base_logvars, base_zs, base_predictions, originals)

In [ ]:
def plot_dimension_analysis(reconstruction_impact, perturbation_sensitivity, kl_divergence, save_path=None):
    """Plot the dimension importance analysis results."""
    n_dims = len(reconstruction_impact)

    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    # Plot reconstruction impact
    axes[0].bar(range(n_dims), reconstruction_impact)
    axes[0].set_title('Reconstruction Impact by Dimension')
    axes[0].set_xlabel('Dimension Index')
    axes[0].set_ylabel('Normalized Impact')

    # Plot perturbation sensitivity
    axes[1].bar(range(n_dims), perturbation_sensitivity)
    axes[1].set_title('Perturbation Sensitivity by Dimension')
    axes[1].set_xlabel('Dimension Index')
    axes[1].set_ylabel('Normalized Sensitivity')

    # Plot KL divergence
    axes[2].bar(range(n_dims), kl_divergence)
    axes[2].set_title('KL Divergence by Dimension')
    axes[2].set_xlabel('Dimension Index')
    axes[2].set_ylabel('Normalized KL Divergence')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
plot_dimension_analysis(reconstruction_impact, perturbation_sensitivity, kl_divergence)

In [ ]:
latent_dims = [2, 4, 6, 8, 16, 32, 64, 128]

mean_mse = []
mean_psnr = []
mean_ssim = []

min_reconstruction_impact = []
min_perturbation_sensitivity = []
min_kl_divergence = []

for dim in latent_dims:
  latent_dim = dim

  model_path = f'./data/VAE_model_weights/cvae_{latent_dim}d_10000_boxbounded.weights.h5'

  model = CVAE(latent_dim)
  model.build(input_shape=(256, 256, 1))
  model.load_weights(model_path)

  base_means, base_logvars, base_zs, base_predictions, originals, mse_values, psnr_values, ssim_values = generate_base_predictions(model)

  reconstruction_impact, perturbation_sensitivity, kl_divergence = analyze_dimension_importance(model, base_means, base_logvars, base_zs, base_predictions, originals)

  mean_mse.append(np.mean(mse_values))
  mean_psnr.append(np.mean(psnr_values))
  mean_ssim.append(np.mean(ssim_values))

  min_reconstruction_impact.append(np.min(reconstruction_impact))
  min_perturbation_sensitivity.append(np.min(perturbation_sensitivity))
  min_kl_divergence.append(np.min(kl_divergence))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

# Plot each metric
plt.plot(latent_dims, mean_ssim, 'o-', label='Mean SSIM')
plt.plot(latent_dims, min_reconstruction_impact, 'o-', label='Min Reconstruction Impact')
plt.plot(latent_dims, min_perturbation_sensitivity, 'o-', label='Min Perturbation Sensitivity')
plt.plot(latent_dims, min_kl_divergence, 'o-', label='Min KL Divergence')

# Set x-axis to log scale since latent_dims spans multiple orders of magnitude
plt.xscale('log')

# Add labels and title
plt.xlabel('Latent Dimensions')
plt.ylabel('Metrics')
plt.title('Metrics vs Latent Dimensions')

# Add legend
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Add grid for better readability
plt.grid(True, which="both", ls="-", alpha=0.2)

# Adjust layout to prevent label cutoff
plt.tight_layout()

plt.show()

In [ ]:
# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot MSE
ax1.plot(latent_dims, mean_mse, 'o-', color='blue')
ax1.set_xscale('log')
ax1.set_xlabel('Latent Dimensions')
ax1.set_ylabel('Mean MSE')
ax1.set_title('Mean MSE vs Latent Dimensions')
ax1.grid(True, alpha=0.2)

# Plot PSNR
ax2.plot(latent_dims, mean_psnr, 'o-', color='green')
ax2.set_xscale('log')
ax2.set_xlabel('Latent Dimensions')
ax2.set_ylabel('Mean PSNR')
ax2.set_title('Mean PSNR vs Latent Dimensions')
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
dimension_analysis_results_df = pd.DataFrame({
    'latent_dims': latent_dims,
    'mean_mse': mean_mse,
    'mean_psnr': mean_psnr,
    'mean_ssim': mean_ssim,
    'min_reconstruction_impact': min_reconstruction_impact,
    'min_perturbation_sensitivity': min_perturbation_sensitivity,
    'min_kl_divergence': min_kl_divergence
})

dimension_analysis_results_df

In [ ]:
dimension_analysis_results_df.to_csv('dimension_analysis_results.csv', index=False)